# 2주차 4교시 · 30일 활동 기록을 회원 한 명당 한 행으로 요약하기

**학생용 지적실험노트**

오늘의 질문: **회원마다 30줄씩 있는 활동 기록을 어떻게 한 줄로 줄일까요?**

오늘의 완성 결과: 30,000행의 하루 기록을 회원별로 요약하고 기본정보를 붙여 1,000행×16열 표를 저장합니다.

## 시작하기 전에 폴더를 확인하세요

- `activity_batches`는 강사가 ZIP 안에 미리 넣어 준 폴더입니다. 학생이 새로 만들지 않습니다.
- ZIP을 먼저 압축 해제하고, 압축을 풀어서 생긴 최상위 폴더 전체를 VS Code에서 엽니다.
- 이 노트북이나 CSV 파일을 다른 폴더로 따로 옮기지 않습니다.
- 자세한 순서는 `week2/00_학생용_먼저읽기_압축해제와VSCode열기.md`를 확인합니다.

각 Q에서 ① 실행 전 예상 ② ___ 채우기 ③ 실행 ④ 값 하나 바꾸기 ⑤ 결과 이유 한 문장을 지킵니다. 오류 메시지에 ___가 보이면 고장이 아니라 아직 코드 빈칸이 남았다는 뜻입니다.


In [ ]:
from pathlib import Path
import pandas as pd

def find_week2_data_dir():
    """노트북을 어느 교시 폴더에서 열어도 2주차 데이터 폴더를 찾습니다."""
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        candidates = [
            base / "course_materials" / "matchmaking_data" / "02_week2",
            base / "matchmaking_data" / "02_week2",
            base / "02_week2",
        ]
        for candidate in candidates:
            if (candidate / "activity_batches").exists():
                return candidate
    raise FileNotFoundError("02_week2 데이터 폴더를 찾지 못했습니다. python_lecture 프로젝트 안에서 노트북을 열어 주세요.")

DATA_DIR = find_week2_data_dir()
WEEK2_DIR = DATA_DIR.parents[1] / "week2"
OUTPUT_DIR = WEEK2_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print("데이터 폴더:", DATA_DIR)
batch_files = sorted((DATA_DIR / "activity_batches").glob("activity_batch_*.csv"))
frames = []
for file_path in batch_files:
    one_batch = pd.read_csv(file_path)
    one_batch["source_file"] = file_path.name
    frames.append(one_batch)
activity = pd.concat(frames, ignore_index=True)


## Q1. 문자 숫자와 날짜를 계산할 수 있는 새 열로 바꿉니다

**먼저 예상하기**

- 원본 열을 남기고 계산용 새 열을 만드는 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** '미측정' 문자열과 날짜 문자열을 그대로 합산하거나 빼지 않도록 준비합니다.

**현재 자료구조** stay_time_min_num은 float Series, activity_dt와 active_dt는 datetime Series입니다.

**코드 읽기 도움** where는 logged_in이 True인 행에만 날짜를 남깁니다. isna()는 비어 있는 값인지 확인하고, sum()은 그 개수를 셉니다.

**이번에 채울 빈칸** 날짜 변환 함수, 숫자 변환 함수, where, isna를 각각 알맞게 넣습니다.

**정상 결과** 12

**오류가 나면** ValueError 또는 object dtype 유지. errors='coerce'와 새 열 이름을 확인합니다.


In [ ]:
activity["activity_dt"] = pd.___(activity["activity_date"], errors="coerce")
activity["stay_time_min_num"] = pd.___(activity["stay_time_min"], errors="coerce")
activity["active_dt"] = activity["activity_dt"].___(activity["logged_in"])
print(activity["stay_time_min_num"].___().sum())

## Q2. 같은 회원의 30일 기록을 합계·평균·최근 날짜로 요약합니다

**먼저 예상하기**

- 한 회원에게 30줄씩 있는 기록을 한 줄로 만들려면 무엇을 기준으로 묶어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 하루별 로그를 회원별 지표로 바꾸어 모델과 API가 사용하기 쉽게 만듭니다.

**현재 자료구조** member_activity는 member_id별 한 행인 1,000행×9열 DataFrame입니다.

**코드 읽기 도움** True는 합계에서 1, False는 0처럼 계산됩니다. 각 지표에 합계·평균·최댓값 중 어떤 집계 방식을 쓸지 빈칸을 채웁니다.

**이번에 채울 빈칸** 집계 함수 이름과 마지막 출력에서 확인할 DataFrame 이름을 채웁니다.

**정상 결과** (1000, 9)

**오류가 나면** 한 회원이 여러 행으로 남거나 MultiIndex가 생김. groupby 기준이 member_id인지, 끝에 reset_index()가 있는지 확인합니다.


In [ ]:
member_activity = activity.groupby("member_id").___(
    active_days=("logged_in", "___"),
    total_app_opens=("app_opens", "___"),
    total_stay_time_min=("stay_time_min_num", "___"),
    total_profile_views=("profile_views", "___"),
    total_recommendations_viewed=("recommendations_viewed", "___"),
    total_messages_sent=("messages_sent", "___"),
    avg_reply_delay_sec=("avg_reply_delay_sec", "___"),
    last_active_date=("active_dt", "___"),
).reset_index()
print(___.shape)

## Q3. 마지막 로그인 후 며칠이 지났는지 계산합니다

**먼저 예상하기**

- 날짜 자체보다 경과일 숫자가 필요한 이유는 무엇일까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 날짜 자체보다 '마지막 활동 후 며칠 지났는가'를 계산에 바로 쓸 수 있게 만듭니다.

**현재 자료구조** days_since_last_activity는 숫자 Series이며 로그인 기록이 전혀 없는 20명은 NaN입니다.

**코드 읽기 도움** 기준 날짜를 만들고 마지막 활동일을 뺍니다. 날짜 차이에서 .dt.days로 일수만 꺼냅니다. 마지막 출력에서는 결측치 확인 메서드도 채웁니다.

**이번에 채울 빈칸** Timestamp, 마지막 활동 날짜 열 이름, days 변환 속성, 결과 열 이름, 결측치 확인 메서드를 채웁니다.

**정상 결과** 20

**오류가 나면** 음수 또는 AttributeError. 뺄셈 순서와 last_active_date dtype을 확인합니다.


In [ ]:
reference_date = pd.___("2026-07-01")
member_activity["days_since_last_activity"] = (
    reference_date - member_activity["___"]
).dt.___
print(member_activity["___"].___().sum())

## Q4. 회원 기본정보와 활동 요약을 member_id로 붙입니다

**먼저 예상하기**

- 활동 기록이 없는 회원도 남기려면 어느 표를 왼쪽에 두어야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 나이·지역 같은 회원 정보와 30일 활동 지표를 같은 행에서 사용합니다.

**현재 자료구조** members는 1,000행×59열, model_table은 필요한 회원 열과 활동 요약을 합친 1,000행×16열 DataFrame입니다.

**코드 읽기 도움** 두 표를 연결할 기준 열, 왼쪽 표를 보존하는 방식, 중복 관계를 검사하는 설정, 결과 크기를 확인하는 속성을 빈칸으로 채웁니다.

**이번에 채울 빈칸** merge, member_id, left, one_to_one, shape를 알맞은 위치에 넣습니다.

**정상 결과** (1000, 16)

**오류가 나면** MergeError 또는 행 수가 1,000보다 많음. 두 표의 member_id 중복 수를 먼저 확인합니다.


In [ ]:
members = pd.read_csv(DATA_DIR / "members.csv")
member_columns = ["member_id", "gender", "age", "residence_region", "membership_tier", "signup_channel", "meeting_scheduled"]
model_table = members[member_columns].___(
    member_activity, on="___", how=___, validate=___
)
print(model_table.___)

## Q5. 회원 한 명당 한 행인지 검사하고 요약표를 저장합니다

**먼저 예상하기**

- 다음 교시가 믿고 읽을 수 있도록 저장 전에 무엇을 검사해야 할까요?
- 내 예상: ____________________

**왜 이 일을 하나요?** 다음 교시가 같은 검증된 입력표에서 시작하도록 중간 결과를 고정합니다.

**현재 자료구조** 저장 파일은 1,000행×16열 CSV이며 member_id는 중복되지 않습니다.

**코드 읽기 도움** 중간 결과를 저장하면 다음 교시에서 30,000행 집계를 반복하지 않고도 시작할 수 있습니다.

**정상 결과** member_activity_summary.csv와 (1000, 16)

**오류가 나면** AssertionError 또는 PermissionError. 중복 member_id를 출력하고, 저장 파일이 열려 있으면 닫습니다.


In [ ]:
assert model_table["member_id"].___()
assert model_table.shape == ___
summary_path = OUTPUT_DIR / "member_activity_summary.csv"
model_table.___(summary_path, index=False)
print(summary_path, model_table.shape)

## 마무리 확인

- [ ] 오늘 결과가 **30,000행의 하루 기록을 회원별로 요약하고 기본정보를 붙여 1,000행×16열 표를 저장합니다.**와 같은가?
- [ ] 실행 순서를 자기 말로 설명할 수 있는가?
- [ ] 오류가 났을 때 마지막 줄과 직전 중간 결과를 확인했는가?
